# A question-answering assistant for vibration monitoring

This notebook builds a small retrieval-augmented generation (RAG) pipeline that answers questions about vibration monitoring and condition-monitoring products. It reads a handful of web pages (four vendor product pages and four background articles), indexes them, and answers questions using only what it retrieved, showing which source each answer came from.

Everything here runs locally with open models, so no API key is needed. The code lives in the `rag/` package next to this notebook; the notebook walks through each stage and looks at how well it works.

1. Fetch and clean the pages
2. Split them into chunks
3. Embed the chunks and build a search index
4. Check retrieval on a set of test questions
5. Generate answers
6. Follow-up questions and questions the sources cannot answer

In [1]:
import textwrap

import pandas as pd

from rag.chunk import chunk_pages
from rag.generate import LocalGenerator
from rag.index import VectorIndex, load_embedder
from rag.pipeline import Assistant, evaluate_retrieval
from rag.scrape import fetch_all
from rag.sources import EVAL_QUESTIONS, SOURCES

pd.set_option("display.max_colwidth", 90)


def preview(text, width=150):
    """A short excerpt, enough to see what a chunk is about."""
    return textwrap.shorten(text, width=width, placeholder=" ...")

## 1. Fetch and clean the pages

Each page is downloaded once and cached in `data/cache/` (which is not committed, since the text belongs to its publishers). The cleaner keeps headings, paragraphs and list items, and drops scripts, navigation, footers, repeated blocks and fragments of a few words such as button labels.

In [2]:
pages, failures = fetch_all(SOURCES, cache_dir="data/cache")

summary = pd.DataFrame([
    {"source": page["name"], "passages": len(page["passages"]),
     "words": sum(len(p.split()) for p in page["passages"])}
    for page in pages
])
print("Failed to fetch:", failures if failures else "none")
summary

Failed to fetch: none


,source,passages,words
0,Fluke,30,696
1,Erbessd Instruments,17,238
2,Acoem,50,1359
3,Emerson,9,152
4,Wikipedia: Condition monitoring,65,3055
5,Wikipedia: Predictive maintenance,64,2312
6,Wikipedia: Vibration,142,5500
7,Wikipedia: Accelerometer,152,5639


## 2. Split into chunks

Pages are split into chunks of whole sentences, at most 120 words each, with one sentence of overlap between neighbours. Splitting on sentence boundaries only works if the punctuation is still there, so the cleaning step deliberately leaves it in.

In [3]:
records = chunk_pages(pages, max_words=120, overlap_sentences=1)

chunk_words = pd.Series([len(r["text"].split()) for r in records])
print(f"{len(records)} chunks; median {chunk_words.median():.0f} words, longest {chunk_words.max()} words")
pd.Series([r["source"] for r in records]).value_counts().rename("chunks").to_frame()

210 chunks; median 112 words, longest 135 words


,chunks
Wikipedia: Vibration,62
Wikipedia: Accelerometer,61
Wikipedia: Condition monitoring,36
Wikipedia: Predictive maintenance,25
Acoem,14
Fluke,7
Erbessd Instruments,3
Emerson,2


The background articles are much longer than the vendor pages, so they contribute most of the chunks. That imbalance matters for retrieval: a question about a product has to find a few relevant chunks among a few hundred general ones.

## 3. Embed and index

Chunks are embedded with `all-MiniLM-L6-v2`, a small sentence-embedding model (384 dimensions), and kept in a matrix. With a few hundred chunks, exact search is one matrix product per query, so no approximate index is needed. Vectors are normalised, so the search score is cosine similarity: 1 means identical direction, 0 means unrelated.

In [4]:
embedder = load_embedder()
index = VectorIndex(embedder, records)
print(f"Indexed {len(index)} chunks of dimension {index.dimension}")

Indexed 210 chunks of dimension 384


In [5]:
for result in index.search("How do wireless vibration sensors help maintenance teams?", top_k=3):
    print(f"{result['score']:.3f}  [{result['source']}]  {preview(result['text'])}")

0.737  [Fluke]  That way you can foresee and prevent failures months in advance. Diagnostic engine trained on 30 years of 100 trillion data points across 50 ...
0.697  [Fluke]  How do wireless sensors support predictive maintenance strategies?. As wireless sensors transmit equipment data to predictive maintenance software ...
0.631  [Fluke]  Make informed decisions to prevent product defects and downtime. High-resolution data allows detection of more faults Triaxial vibration system ...


## 4. Does retrieval find the right source?

Before looking at generated answers it is worth checking retrieval on its own, because a generator can only be as good as the context it is given. Ten test questions each name the source that should come back. `hit@k` is the share of questions whose expected source appears in the top k chunks.

In [6]:
rows, hit_rates = evaluate_retrieval(index, EVAL_QUESTIONS, ks=(1, 3, 5))
print({name: f"{value:.0%}" for name, value in hit_rates.items()})
pd.DataFrame(rows)[["question", "expected", "rank", "top_source", "top_score"]]

{'hit@1': '90%', 'hit@3': '90%', 'hit@5': '90%'}


,question,expected,rank,top_source,top_score
0,What vibration monitoring tools does Fluke offer?,Fluke,NaN,Erbessd Instruments,0.578
1,What is the Machine Defender app used for?,Acoem,1.0,Acoem,0.697
2,What is the AMS 2140 machinery health analyzer?,Emerson,1.0,Emerson,0.803
3,What products does Erbessd Instruments make?,Erbessd Instruments,1.0,Erbessd Instruments,0.481
4,What is condition monitoring?,Wikipedia: Condition monitoring,1.0,Wikipedia: Condition monitoring,0.823
5,How does predictive maintenance differ from preventive maintenance?,Wikipedia: Predictive maintenance,1.0,Wikipedia: Predictive maintenance,0.837
6,What is the difference between free and forced vibration?,Wikipedia: Vibration,1.0,Wikipedia: Vibration,0.769
7,How does a piezoelectric accelerometer work?,Wikipedia: Accelerometer,1.0,Wikipedia: Accelerometer,0.708
8,What is resonance?,Wikipedia: Vibration,1.0,Wikipedia: Vibration,0.575
9,Which techniques are used to monitor rotating machinery?,Wikipedia: Condition monitoring,1.0,Wikipedia: Condition monitoring,0.709


One question misses completely: "What vibration monitoring tools does Fluke offer?" returns Erbessd chunks, and no Fluke chunk appears in the top five. The reason is visible in the Fluke excerpts above: the page describes its products without repeating the company name in each paragraph, so once the page is cut into chunks, nothing in the text says "Fluke".

A simple fix is to embed each chunk together with the name of its source ("Fluke: ...").

In [7]:
index_named = VectorIndex(embedder, records, include_source=True)
rows_named, hit_rates_named = evaluate_retrieval(index_named, EVAL_QUESTIONS, ks=(1, 3, 5))

comparison = pd.DataFrame({"chunk text only": hit_rates, "source name + chunk text": hit_rates_named})
display((comparison * 100).round(0).astype(int).astype(str) + "%")
pd.DataFrame({
    "question": [r["question"] for r in rows],
    "rank (text only)": [r["rank"] for r in rows],
    "rank (with source name)": [r["rank"] for r in rows_named],
})

,chunk text only,source name + chunk text
hit@1,90%,100%
hit@3,90%,100%
hit@5,90%,100%


,question,rank (text only),rank (with source name)
0,What vibration monitoring tools does Fluke offer?,NaN,1
1,What is the Machine Defender app used for?,1.0,1
2,What is the AMS 2140 machinery health analyzer?,1.0,1
3,What products does Erbessd Instruments make?,1.0,1
4,What is condition monitoring?,1.0,1
5,How does predictive maintenance differ from preventive maintenance?,1.0,1
6,What is the difference between free and forced vibration?,1.0,1
7,How does a piezoelectric accelerometer work?,1.0,1
8,What is resonance?,1.0,1
9,Which techniques are used to monitor rotating machinery?,1.0,1


With the source name included, all ten questions find their source at rank 1. Ten hand-written questions are a small test, so this shows the fix works for the failure that was observed, not that retrieval is solved. The assistant below uses this index.

## 5. Generate answers

The generator is `google/flan-t5-base`, a 250-million-parameter open model that runs on a laptop CPU. It accepts 512 tokens of input, so before each call the lowest-ranked chunks are dropped until the prompt fits. Without that step the tokenizer would cut off the end of the prompt, which is where the question is.

In [8]:
generator = LocalGenerator()
assistant = Assistant(index_named, generator, top_k=4, min_score=0.25)


def show(result):
    print("Q:", result["question"])
    print("A:", result["answer"])
    for chunk in result["sources"]:
        print(f"   {chunk['score']:.3f}  [{chunk['source']}]  {preview(chunk['text'], 110)}")
    print()

In [9]:
questions = [
    "What is condition monitoring?",
    "What is the Machine Defender app used for?",
    "What does the AMS 2140 do?",
    "What causes resonance?",
    "What is predictive maintenance?",
]
for question in questions:
    assistant.reset()
    show(assistant.ask(question))

Q: What is condition monitoring?
A: the process of monitoring a parameter of condition in machinery (vibration, temperature etc.), in order to identify a significant change which is indicative of a developing fault.
   0.792  [Wikipedia: Condition monitoring]  Condition monitoring (colloquially, CM ) is the process of monitoring a parameter of condition in ...
   0.704  [Wikipedia: Condition monitoring]  These systems are designed for permanent installation as a condition monitoring solution rather than as a ...
   0.700  [Wikipedia: Condition monitoring]  The concept of this type of device is that it can be used by normal plant operators and maintainers ...



Q: What is the Machine Defender app used for?
A: Tablet based instant machinery diagnostic. Gain a better understanding of machine diagnostics and the health of your rotating assets quickly, easily and accurately using the latest version of the Machine Defender app in conjunction with a wireless sensor.
   0.658  [Acoem]  Machine Defender App. Tablet based instant machinery diagnostic. Gain a better understanding of machine ...
   0.582  [Acoem]  Resources. RT-300 - Machine Defender Application - Diagnosis. The RT-300 is the preventive maintenance ...
   0.576  [Acoem]  Acoem Machine Defender app has been designed to put rotating machinery health assessment at the grasp of ...



Q: What does the AMS 2140 do?
A: takes vibration data and analysis measurements to the next level
   0.573  [Emerson]  AMS 2140 Machinery Health Analyzer. The AMS 2140 Machinery Health Analyzer takes vibration data and ...
   0.336  [Wikipedia: Accelerometer]  Scotland's Queensferry Crossing was equipped with a comprehensive SHM system including thousands of ...



Q: What causes resonance?
A: When the forcing frequency nears the natural frequency ( ) the amplitude of the vibration can get extremely high. This phenomenon is called resonance (subsequently the natural frequency of a system is often referred to as the resonant frequency).
   0.557  [Wikipedia: Vibration]  If resonance occurs in a mechanical system it can be very harmful – leading to eventual failure of the ...
   0.540  [Wikipedia: Vibration]  The phase shift, is defined by the following formula. The plot of these functions, called "the frequency ...
   0.458  [Wikipedia: Vibration]  Whatever the damping is, when r ≫ 1, the vibration is 180 degrees out of phase with the forcing frequency ...



Q: What is predictive maintenance?
A: evaluates the condition of equipment by performing periodic (offline) or continuous (online) equipment condition monitoring . The ultimate goal of the approach is to perform maintenance at a scheduled point in time when the maintenance activity is most cost-effective and before the equipment loses performance within a threshold.
   0.848  [Wikipedia: Predictive maintenance]  Predictive maintenance evaluates the condition of equipment by performing periodic (offline) or continuous ...
   0.827  [Wikipedia: Predictive maintenance]  Predictive maintenance ( PdM ) techniques are designed to help determine the condition of in-service ...
   0.801  [Wikipedia: Predictive maintenance]  The main appeal of predictive maintenance is to allow convenient scheduling of corrective maintenance , ...



## 6. Follow-ups and unanswerable questions

The assistant keeps the conversation history. For a follow-up, the previous question is added to the search query, so "Who makes it?" is searched together with the question before it.

In [10]:
assistant.reset()
show(assistant.ask("What is the Machine Defender app used for?"))
show(assistant.ask("Which company makes it?"))

Q: What is the Machine Defender app used for?
A: Tablet based instant machinery diagnostic. Gain a better understanding of machine diagnostics and the health of your rotating assets quickly, easily and accurately using the latest version of the Machine Defender app in conjunction with a wireless sensor.
   0.658  [Acoem]  Machine Defender App. Tablet based instant machinery diagnostic. Gain a better understanding of machine ...
   0.582  [Acoem]  Resources. RT-300 - Machine Defender Application - Diagnosis. The RT-300 is the preventive maintenance ...
   0.576  [Acoem]  Acoem Machine Defender app has been designed to put rotating machinery health assessment at the grasp of ...



Q: Which company makes it?
A: Acoem
   0.654  [Acoem]  Machine Defender App. Tablet based instant machinery diagnostic. Gain a better understanding of machine ...
   0.592  [Acoem]  Resources. RT-300 - Machine Defender Application - Diagnosis. The RT-300 is the preventive maintenance ...



When nothing in the index is close to the question (top similarity below 0.25), the assistant declines without calling the generator at all. This is a cheap guard against a model inventing an answer from unrelated context.

In [11]:
assistant.reset()
for question in ["What is the best recipe for sourdough bread?", "Who won the 2018 football World Cup?"]:
    top = index_named.search(question, top_k=1)[0]
    print(f"Top similarity {top['score']:.3f}")
    show(assistant.ask(question))

Top similarity 0.110
Q: What is the best recipe for sourdough bread?
A: I don't know based on the sources I have.

Top similarity 0.091
Q: Who won the 2018 football World Cup?
A: I don't know based on the sources I have.



## What this shows

- **Retrieval is the part that matters most.** With the right chunk at the top, even a small model produces a correct answer. When the Fluke question retrieved another vendor's chunks, no generator could have fixed it.
- **Chunks lose their context.** A paragraph cut out of a product page no longer says whose product it describes. Embedding the source name with each chunk took the test set from 9 of 10 to 10 of 10 at rank 1.
- **Cleaning has to preserve sentence boundaries.** Stripping all punctuation before splitting into sentences leaves nothing to split on, and every page collapses into a single chunk.
- **The local model mostly extracts.** `flan-t5-base` tends to copy a sentence or two from the context. That makes its answers easy to verify against the listed sources, but it does not compare or summarise across them. Setting `GROQ_API_KEY` or `OPENAI_API_KEY` switches to a hosted chat model through the same interface.
- **Refusing is cheap.** A similarity threshold on the best match stops off-topic questions before they reach the generator.

Limits: the test set is ten questions written by hand, the similarity threshold of 0.25 was chosen by looking at a few examples, and vendor pages change, so results will drift as the sites are updated.